# LangGraph com AgentCore Memory Hooks (Memória de Longo Prazo)

## Introdução

Este notebook demonstra como integrar as capacidades do Amazon Bedrock AgentCore Memory com um agente de IA conversacional usando o framework LangGraph. Vamos focar na retenção de **memória de longo prazo** entre múltiplas sessões de conversa - permitindo que um agente extraia e recupere preferências do usuário, restrições alimentares e informações contextuais de interações anteriores.

## Detalhes do Tutorial

| Informação                | Detalhes                                                                         |
|:--------------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial          | Conversacional de Longo Prazo                                                   |
| Caso de uso do agente     | Assistente de Nutrição                                                           |
| Framework de agentes      | LangGraph                                                                        |
| Modelo LLM                | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial   | AgentCore Long-term Memory, Custom Memory Strategies, Pre/Post Model Hooks      |
| Complexidade do exemplo   | Intermediário                                                                    |

Você aprenderá a:
- Criar AgentCore Memory com a estratégia custom-override UserPreference
- Implementar hooks pre/post model para armazenamento e recuperação automática de memória
- Construir um assistente de nutrição que lembra as preferências do usuário entre sessões
- Usar busca semântica para recuperar contexto relevante do usuário
- Configurar prompts personalizados de extração e consolidação de memória

### Contexto do Cenário

Neste exemplo, criaremos um **Assistente de Nutrição** que pode lembrar o contexto do usuário entre múltiplas conversas, incluindo restrições alimentares, alimentos favoritos, preferências culinárias e metas de saúde. O agente extrairá e armazenará automaticamente as preferências do usuário a partir das conversas, e depois recuperará o contexto relevante para interações futuras, fornecendo aconselhamento nutricional personalizado.

## Arquitetura

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Pré-requisitos

- Python 3.10+
- Conta AWS com permissões apropriadas
- IAM role da AWS com permissões apropriadas para AgentCore Memory
- Acesso aos modelos do Amazon Bedrock

Vamos começar configurando nosso ambiente!

In [ ]:
# Install necessary libraries from https://github.com/langchain-ai/langchain-aws
%pip install -qr requirements.txt

In [ ]:
import os
import logging

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid


region = os.getenv("AWS_REGION", "us-east-1")
logging.getLogger("math-agent").setLevel(logging.DEBUG)

In [ ]:
# Import the AgentCoreMemoryStore that we will use as a store
from langgraph_checkpoint_aws import AgentCoreMemoryStore

# For this example, we will just use an InMemorySaver to save context.
# In production, we highly recommend the AgentCoreMemorySaver as a checkpointer which works seamlessly alongside the memory store
# from langgraph_checkpoint_aws import AgentCoreMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from custom_memory_prompts import consolidation_prompt, extraction_prompt

In [ ]:
memory_name = "NutritionAssistant"
client = MemoryClient(region_name=region)
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant",
    memory_execution_role_arn="arn:aws:iam::YOUR_ACCOUNT:role/YOUR_ROLE",  # Please provide a role with a valid trust policy
    strategies=[
        {
            StrategyType.CUSTOM.value: {
                "name": "NutritionPreferences",
                "description": "Captures customer food preferences and behavior",
                "namespaceTemplates": ["/{actorId}/preferences/"],
                "configuration": {
                    "userPreferenceOverride": {
                        "extraction": {
                            "appendToPrompt": extraction_prompt,
                            "modelId": MODEL_ID,
                        },
                        "consolidation": {
                            "appendToPrompt": consolidation_prompt,
                            "modelId": MODEL_ID,
                        },
                    }
                },
            }
        },
    ],
)
memory_id = memory["id"]

### Visão Geral da Configuração de Memória

Nossa configuração do AgentCore Memory inclui:

- **Custom Strategy**: Extrai preferências nutricionais das conversas
- **Namespaces**: Organiza memórias por usuário (`{actorId}/preferences/`)
- **Custom Prompts**: Lógica especializada de extração e consolidação para preferências alimentares
- **Integração com Modelo**: Usa Claude 3.7 Sonnet para processamento de memória

O sistema de memória processará automaticamente as conversas para extrair preferências duradouras do usuário, filtrando informações temporárias ou irrelevantes.

## Passo 3: Inicializar o Memory Store e o LLM

Agora vamos inicializar o AgentCore Memory Store e nosso modelo de linguagem.

In [ ]:
# Initialize the store to enable long term memory saving and retrieval
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Initialize Bedrock LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

## Passo 4: Implementar Memory Hooks

Criaremos hooks pre e post model para lidar automaticamente com o armazenamento e recuperação de memória:

- **Pre-model hook**: Recupera preferências relevantes do usuário (baseado em busca semântica) e adiciona contexto antes da invocação do LLM
- **Post-model hook**: Salva as mensagens da conversa para extração de memória de longo prazo

### Como o Processamento de Memória Funciona

1. As mensagens são salvas no AgentCore Memory com actor_id e session_id
2. A custom strategy processa as conversas para extrair preferências nutricionais
3. As preferências extraídas são armazenadas no namespace `{actorId}/preferences/`
4. Conversas futuras podem buscar e recuperar preferências relevantes para contexto

**Nota**: Os tipos de mensagem do LangChain são convertidos internamente pelo store para tipos de mensagem do AgentCore Memory, para que possam ser devidamente extraídos como memórias de longo prazo.

In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs pre-LLM invocation to save the latest human message"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # Save the last human message we see before LLM invocation
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break
    # Retrieve user preferences based on the last message and append to state
    user_preferences_namespace = (actor_id, "preferences/")
    preferences = store.search(user_preferences_namespace, query=msg.content, limit=5)

    # Construct another AI message to add context before the current message
    if preferences:
        context_items = [pref.value for pref in preferences]
        context_message = AIMessage(
            content=f"[User Context: {', '.join(str(item) for item in context_items)}]"
        )
        # Insert the context message before the last human message
        return {"messages": messages[:-1] + [context_message, messages[-1]]}

    return {"llm_input_messages": messages}


def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs post-LLM invocation to save the latest human message"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # Save the LLMs response to AgentCore Memory
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    return {"messages": messages}

## Passo 5: Criar o Agente LangGraph

Agora criaremos nosso agente assistente de nutrição usando o `create_react_agent` do LangGraph com nossos memory hooks integrados. O tool node conterá apenas nossa ferramenta de recuperação de memória de longo prazo, e os hooks pre e post model são especificados como argumentos.

**Nota**: para implementações customizadas de agentes, o Store e as tools podem ser configurados para executar conforme necessário para qualquer workflow seguindo este padrão. Hooks pre/post model podem ser usados, toda a conversa pode ser salva no final, etc.

In [ ]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[],  # No additional tools needed for this example
    checkpointer=InMemorySaver(),  # For conversation state management
    pre_model_hook=pre_model_hook,  # Retrieves user preferences before LLM call
    post_model_hook=post_model_hook,  # Saves conversation after LLM response
)

## Passo 6: Configurar o Runtime do Agente

Precisamos configurar o agente com identificadores únicos para o usuário e a sessão. Esses IDs são cruciais para a organização e recuperação de memória.

### Entrada de Invocação do Graph
Precisamos apenas passar a mensagem mais recente do usuário como argumento `inputs`. Isso poderia incluir outras variáveis de estado também, mas para o simples `create_react_agent`, precisamos apenas de messages.

### LangGraph RuntimeConfig
No LangGraph, config é um `RuntimeConfig` que contém atributos necessários no momento da invocação, por exemplo IDs de usuário ou IDs de sessão. Para o `AgentCoreMemorySaver`, `thread_id` e `actor_id` devem ser definidos no config. Por exemplo, seu endpoint de invocação do AgentCore poderia atribuir isso com base na identidade ou ID do usuário do chamador. Você pode ler a [documentação adicional aqui](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)



In [ ]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1",  # REQUIRED: This maps to Bedrock AgentCore session_id under the hood
        "actor_id": actor_id,  # REQUIRED: This maps to Bedrock AgentCore actor_id under the hood
    }
}

## Passo 7: Testar o Agente

Vamos testar nosso assistente de nutrição tendo uma conversa sobre preferências alimentares. O agente extrairá e armazenará automaticamente as preferências do usuário para uso futuro.

In [ ]:
# Helper function to pretty print agent output while running
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # Check if we've already printed this message
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)

### O que foi armazenado?
Como você pode ver, o modelo ainda não tem nenhuma informação sobre nossas preferências ou restrições alimentares.

Para esta implementação com hooks pre/post model, duas mensagens foram armazenadas aqui. A primeira mensagem do usuário e a resposta do modelo de IA foram ambas armazenadas como eventos conversacionais no AgentCore Memory. Pode levar alguns instantes para que as memórias de longo prazo sejam extraídas, então tente novamente após alguns segundos se nada for encontrado na primeira tentativa.

Essas mensagens foram então extraídas para a memória de longo prazo do AgentCore nos nossos namespaces de fatos e preferências do usuário. Na verdade, podemos verificar o store nós mesmos para conferir o que foi armazenado até agora:

In [ ]:
# Search our user preferences namespace
search_namespace = (actor_id, "preferences/")
result = store.search(search_namespace, query="food", limit=3)
print(f"Preferences namespace result: {result}")

### Acesso do agente ao store

**Nota** - como o AgentCore Memory processa esses eventos em segundo plano, pode levar alguns segundos para que a memória seja extraída e incorporada à recuperação de memória de longo prazo.

Ótimo! Agora vimos que memórias de longo prazo foram extraídas para nossos namespaces com base nas mensagens anteriores da conversa.

Agora, vamos iniciar uma nova sessão e perguntar sobre recomendações do que cozinhar para o jantar. O agente pode usar o store para acessar as memórias de longo prazo que foram extraídas para fazer uma recomendação que o usuário certamente irá gostar.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # New session ID
        "actor_id": actor_id,  # Same actor ID
    }
}

run_agent("Today's a new day, what should I make for dinner tonight?", config)

### Conclusão

Como você pode ver, o agente recebeu contexto do pre-model hook a partir da busca no namespace de preferências do usuário e também foi capaz de buscar por conta própria memórias de longo prazo no namespace de fatos para criar uma resposta abrangente para o usuário.

O AgentCoreMemoryStore é muito flexível e pode ser implementado de várias maneiras, incluindo hooks pre/post model ou apenas tools com operações de store. Usado em conjunto com o AgentCoreMemorySaver para checkpointing, tanto o estado conversacional completo quanto insights de longo prazo podem ser combinados para formar um sistema de agentes complexo e inteligente.